# 03 - Features, linear regression and charts

Input: `data/processed/PL_2025_26_model_data.csv` (from notebook 02).

Plan:
1. Look at the target (market value is heavily skewed, so we model its log)
2. Explore the main value drivers with a few charts
3. Train a baseline **linear regression** and evaluate it with R² and MAE
4. Compare with a simple baseline and (as a "next step") a random forest
5. Save the charts to `figures/` for the slides

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

DATA = Path("../data/processed/PL_2025_26_model_data.csv")
FIG = Path("../figures"); FIG.mkdir(exist_ok=True)

# Chart styling: one color for single-series charts, hairline solid gridlines, quiet axes
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE = "#2a78d6"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "axes.axisbelow": True, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "regular", "axes.titlelocation": "left",
})

def eur(v, _=None):
    if v >= 1e6:
        return f"€{v/1e6:g}M"
    return f"€{v/1e3:g}K"
EUR = FuncFormatter(eur)

df = pd.read_csv(DATA)
df["log_value"] = np.log(df["market_value"])
print(df.shape)
df.head()

## 1. The target: market value is skewed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

axes[0].hist(df["market_value"] / 1e6, bins=30, color=BLUE, edgecolor=SURFACE, linewidth=1.2)
axes[0].set_title("Market value (€M)")
axes[0].set_xlabel("€ millions"); axes[0].set_ylabel("players")

bins = np.logspace(np.log10(df["market_value"].min()), np.log10(df["market_value"].max()), 30)
axes[1].hist(df["market_value"], bins=bins, color=BLUE, edgecolor=SURFACE, linewidth=1.2)
axes[1].set_xscale("log"); axes[1].xaxis.set_major_formatter(EUR)
axes[1].set_title("Same data on a log scale")
axes[1].set_xlabel("market value (log scale)")

fig.tight_layout()
fig.savefig(FIG / "01_value_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print("skew raw:", round(df["market_value"].skew(), 2), "| skew of log:", round(df["log_value"].skew(), 2))

## 2. What drives value? Two quick looks

In [ ]:
def label_top(ax, data, xcol, n=3):
    # label only the few most valuable players; stagger the offsets so labels do not collide
    offsets = [(8, 6, "left"), (8, -14, "left"), (-8, 6, "right")]
    for (_, r), (dx, dy, ha) in zip(data.nlargest(n, "market_value").iterrows(), offsets):
        ax.annotate(r["name"], (r[xcol], r["market_value"]), xytext=(dx, dy), ha=ha,
                    textcoords="offset points", fontsize=8.5, color=INK2)

fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.scatter(df["age"], df["market_value"], s=26, color=BLUE, alpha=0.8, edgecolor=SURFACE, linewidth=0.7)
ax.set_yscale("log"); ax.yaxis.set_major_formatter(EUR)
ax.set_title("Value by age: it peaks in the mid-20s")
ax.set_xlabel("age at valuation date"); ax.set_ylabel("market value (log scale)")
label_top(ax, df, "age")
fig.tight_layout()
fig.savefig(FIG / "02_value_vs_age.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
att = df[df["position"] == "Attack"]
if len(att) < 30:      # fall back to everyone if the label differs
    att = df
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.scatter(att["goals_per90"], att["market_value"], s=26, color=BLUE, alpha=0.8, edgecolor=SURFACE, linewidth=0.7)
ax.set_yscale("log"); ax.yaxis.set_major_formatter(EUR)
ax.set_title("Attackers: value rises with goals per 90")
ax.set_xlabel("goals per 90 minutes (2025/26)"); ax.set_ylabel("market value (log scale)")
label_top(ax, att, "goals_per90")
fig.tight_layout()
fig.savefig(FIG / "03_value_vs_goals.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
num_cols = ["age", "minutes", "goals_per90", "assists_per90", "international_caps", "contract_years_left"]
corr = df[num_cols].corrwith(df["log_value"]).sort_values(ascending=False).round(2).to_frame("correlation with log value")
corr

## 3. Linear regression

- Target: `log(market value)`
- Features: age, age² (value is not linear in age), minutes, goals and assists per 90, international caps, contract years left, position (one-hot)
- Missing contract years are filled with the median
- 80/20 train/test split, plus 5-fold cross-validation for a steadier estimate

In [ ]:
df["age_sq"] = df["age"] ** 2
num_features = ["age", "age_sq", "minutes", "goals_per90", "assists_per90",
                "international_caps", "contract_years_left"]
cat_features = ["position"]
X = df[num_features + cat_features]
y = df["log_value"]

prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
])

def make(model):
    return Pipeline([("prep", prep), ("model", model)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("train:", len(X_train), "| test:", len(X_test))

In [ ]:
def evaluate(name, pipe):
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    cv = cross_val_score(make(pipe.named_steps["model"]), X, y,
                         cv=KFold(5, shuffle=True, random_state=42), scoring="r2")
    return {
        "model": name,
        "R² (test, log value)": round(r2_score(y_test, pred), 3),
        "R² (5-fold CV)": round(cv.mean(), 3),
        "MAE (test, €M)": round(mean_absolute_error(np.exp(y_test), np.exp(pred)) / 1e6, 2),
        "median abs % error": round(float(np.median(np.abs(np.exp(pred) / np.exp(y_test) - 1))) * 100, 1),
    }

lin = make(LinearRegression())
results = [
    evaluate("Baseline (predict the average)", make(DummyRegressor())),
    evaluate("Linear regression", lin),
]
pd.DataFrame(results).set_index("model")

**How to read this:** R² is measured on the log of the value, so it says how much of the *relative* differences between players the model explains. MAE is in euros after converting predictions back with `exp`. The baseline row shows what "no model" scores, so the linear regression's improvement is visible.

In [ ]:
pred_test = lin.predict(X_test)
lo, hi = np.exp(min(y_test.min(), pred_test.min())), np.exp(max(y_test.max(), pred_test.max()))

fig, ax = plt.subplots(figsize=(6, 5.4))
ax.scatter(np.exp(y_test), np.exp(pred_test), s=26, color=BLUE, alpha=0.8, edgecolor=SURFACE, linewidth=0.7)
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1)          # perfect prediction line
ax.set_xscale("log"); ax.set_yscale("log")
ax.xaxis.set_major_formatter(EUR); ax.yaxis.set_major_formatter(EUR)
ax.set_xlabel("actual market value"); ax.set_ylabel("predicted market value")
ax.set_title("Predicted vs actual (test players)")
ax.text(0.04, 0.93, f"R² = {r2_score(y_test, pred_test):.2f}", transform=ax.transAxes, color=INK2)
fig.tight_layout()
fig.savefig(FIG / "04_predicted_vs_actual.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. What does the model say matters?

In [ ]:
names = lin.named_steps["prep"].get_feature_names_out()
all_coefs = pd.Series(lin.named_steps["model"].coef_, index=[n.split("__")[1] for n in names])

# Age enters as age and age² (a curve), so those two coefficients cancel each other and are not
# readable alone - the age chart above shows the age effect. Position dummies are relative effects.
# The chart shows the other numeric drivers, which are standardized (per 1 standard deviation).
shown = all_coefs.drop(["age", "age_sq"]).filter(regex="^(?!position_)").sort_values()

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.barh(shown.index, shown.values, color=BLUE, height=0.5)
ax.axvline(0, color=AXIS, linewidth=1)
ax.set_title("Other drivers: effect on log value of +1 standard deviation")
ax.set_xlabel("coefficient (log value)")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIG / "05_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()
all_coefs.round(3).to_frame("coefficient").sort_values("coefficient", ascending=False)

## 5. Biggest disagreements with Transfermarkt

Out-of-sample predictions for every player (each prediction comes from a model that did not see that player). A large gap is not proof of a mistake - the model ignores club, injuries and transfer rumours - but it is a useful sanity check and a good discussion point.

In [ ]:
oof = cross_val_predict(make(LinearRegression()), X, y, cv=KFold(5, shuffle=True, random_state=42))
out = df[["name", "current_club_name", "position", "age", "market_value"]].copy()
out["predicted_value"] = np.exp(oof)
out["pred / actual"] = (out["predicted_value"] / out["market_value"]).round(2)
cols = ["name", "current_club_name", "position", "age", "market_value", "predicted_value", "pred / actual"]
print("Model says worth MORE than Transfermarkt:")
display(out.sort_values("pred / actual", ascending=False).head(8)[cols].round(1))
print("Model says worth LESS than Transfermarkt:")
display(out.sort_values("pred / actual").head(8)[cols].round(1))

## 6. Next step preview: a random forest on the same features

In [ ]:
rf = evaluate("Random forest (next step)", make(RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1)))
pd.DataFrame(results + [rf]).set_index("model")

## Findings to put in the presentation

- Dataset size and how many players were lost at each step (notebook 02 funnel)
- Baseline vs linear regression: R² and MAE from the table above
- The strongest drivers (coefficient chart) and whether they match intuition (age curve, goals per 90, caps)
- Limitations: one season, no club strength feature, no injury or contract-negotiation information, fixed data snapshot
- Next steps: add club strength, try the random forest properly, tune, write the final report